In [ ]:
import pandas as pd

def evaluar_dataset_ruido(path_csv):
    """
    Evalúa un dataset de ruido de forma automática.
    Calcula la precisión global del sistema (incluyendo omitidos) 
    y la precisión pura del LLM (solo sobre los logs que analizó).
    """
    df = pd.read_csv(path_csv)
    total_filas_csv = len(df)
    
    omitidos = 0
    aciertos_llm_info_bajo = 0
    falsos_positivos_medio_alto = 0
    errores_parseo = 0
    
    for index, row in df.iterrows():
        # 1. Buscamos la columna estrictamente como 'Justificacion'
        justificacion = str(row.get('Justificacion', '')).strip()
        
        # 2. Verificamos si es un evento omitido por la whitelist
        if 'Omitido' in justificacion:
            omitidos += 1
            continue 
            
        # 3. Si NO fue omitido, leemos qué dijo el LLM
        prediccion_llm = str(row['Riesgo']).upper().strip()
        
        if 'ERROR' in prediccion_llm or 'FALLO' in prediccion_llm:
            errores_parseo += 1
            continue
            
        # 4. Evaluamos el rendimiento del LLM sobre lo que sí procesó
        if prediccion_llm in ['INFO', 'BAJO', 'INFORMACIÓN']:
            aciertos_llm_info_bajo += 1
        elif prediccion_llm in ['MEDIO', 'ALTO']:
            falsos_positivos_medio_alto += 1
            
    # --- CÁLCULOS DE PRECISIÓN ---
    
    # A) Métrica Pura del LLM (Solo lo que el LLM intentó clasificar)
    total_evaluado_llm = aciertos_llm_info_bajo + falsos_positivos_medio_alto
    precision_llm_pura = (aciertos_llm_info_bajo / total_evaluado_llm) * 100 if total_evaluado_llm > 0 else 0
    
    # B) Métrica Global del Pipeline (Whitelist + LLM)
    aciertos_totales_pipeline = aciertos_llm_info_bajo + omitidos
    total_eventos_pipeline = total_evaluado_llm + omitidos
    precision_global = (aciertos_totales_pipeline / total_eventos_pipeline) * 100 if total_eventos_pipeline > 0 else 0
    
    # --- REPORTE ---
    print("="*65)
    print(f"ANÁLISIS DE RUIDO (Especificidad) - Archivo: {path_csv}")
    print("="*65)
    print(f"Total de filas en el CSV       : {total_filas_csv}")
    print(f"Acierto sin omitidos           : {aciertos_totales_pipeline}")
    print(f"Eventos filtrados (Omitidos)   : {omitidos} (Acierto de la Whitelist)")
    print("-" * 65)
    print("RENDIMIENTO DEL LLM (Sobre los eventos que SÍ analizó):")
    print(f"  ✅ Aciertos (INFO/BAJO)             : {aciertos_llm_info_bajo}")
    print(f"  ❌ Falsos Positivos (MEDIO/ALTO)    : {falsos_positivos_medio_alto}")
    if errores_parseo > 0:
        print(f"  ⚠️ Errores de parseo LLM            : {errores_parseo}")
    print("-" * 65)
    print(f"1️⃣  PRECISIÓN PURA DEL LLM         : {precision_llm_pura:.2f}%")
    print(f"2️⃣  PRECISIÓN GLOBAL DEL SISTEMA   : {precision_global:.2f}%")
    print("="*65)
    
    return precision_llm_pura, precision_global

# --- EJECUCIÓN ---
evaluar_dataset_ruido('../results/ruido_resultados_raw_reducido.csv')

ANÁLISIS DE RUIDO (Especificidad) - Archivo: ../results/ruido_resultados_raw_reducido.csv
Total de filas en el CSV       : 475
Acierto sin omitidos: 388
Eventos filtrados (Omitidos)   : 231 (Acierto de la Whitelist)
-----------------------------------------------------------------
RENDIMIENTO DEL LLM (Sobre los eventos que SÍ analizó):
  ✅ Aciertos (INFO/BAJO)             : 157
  ❌ Falsos Positivos (MEDIO/ALTO)    : 87
-----------------------------------------------------------------
1️⃣  PRECISIÓN PURA DEL LLM         : 64.34%
2️⃣  PRECISIÓN GLOBAL DEL SISTEMA   : 81.68%


(64.34426229508196, 81.6842105263158)